# SentinelAI — Explainable AI (XAI) with SHAP
## High-Throughput Explainability for Autonomous Cyber Defense

This notebook implements **SHapley Additive exPlanations (SHAP)** across SentinelAI's **3-Tier AI Pipeline**:
* **Tier 1**: Unsupervised Behavioral Anomaly Detection (`IsolationForest` tree path deviance)
* **Tier 2**: High-Speed Binary Gatekeeper (`LightGBM` log-odds attribution)
* **Tier 3**: Multi-Class Threat Attribution (`LightGBM` 9-family MITRE classification)

**Objective**: Move beyond static black-box predictions to provide Security Operations Center (SOC) analysts
with clear mathematical and visual explanations of **why** a flow was blocked and **which network features** drove the verdict.

### Step 1: Environment Setup & Library Imports

In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import shap

print(f'SHAP Version: {shap.__version__}')
shap.initjs()

### Step 2: Load Trained Production Models & Canonical Schema

In [2]:
TIER1_PATH = '../saved_models/production/tier1_anomaly_detector.joblib'
TIER2_PATH = '../saved_models/production/tier2_binary_gatekeeper.joblib'
TIER3_PATH = '../saved_models/production/tier3_threat_classifier.joblib'
SCALER_PATH = '../saved_models/production/feature_scaler.joblib'
ENCODER_PATH = '../saved_models/production/label_encoder.joblib'

t1_payload = joblib.load(TIER1_PATH)
t1_model = t1_payload['model']
t1_thresh = float(t1_payload['threshold'])
scaler = joblib.load(SCALER_PATH)

t2_booster = joblib.load(TIER2_PATH)
t3_booster = joblib.load(TIER3_PATH)
label_encoder = joblib.load(ENCODER_PATH)
classes = list(label_encoder.classes_)

FEATURE_COLS = [
    'dst_port', 'flow_duration_s', 'flow_iat_mean_s', 'fwd_iat_mean_s', 'bwd_iat_mean_s',
    'fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes', 'fwd_pkt_len_mean',
    'bwd_pkt_len_mean', 'flow_bytes_per_sec', 'flow_pkts_per_sec', 'fwd_tcp_win_ratio',
    'bwd_tcp_win_ratio', 'traffic_symmetry_ratio', 'pkt_len_ratio', 'bytes_per_packet'
]
print(f'Loaded all 3 tiers with {len(classes)} MITRE attack categories and {len(FEATURE_COLS)} features.')

### Step 3: Extract 10,000-Flow Stratified Held-Out Evaluation Dataset

In [3]:
DATA_PATH = '../data/processed/sentinel_20M_transformed.parquet'
pf = pq.ParquetFile(DATA_PATH)

# Stratified sampling across Benign and all 8 attack classes
benign_chunks = []
attack_chunks = {c: [] for c in classes if c != 'Normal'}
cols = FEATURE_COLS + ['label', 'attack_family', 'source_dataset']

for rg in range(pf.metadata.num_row_groups):
    df_c = pf.read_row_group(rg, columns=cols).to_pandas()
    if len(benign_chunks) < 5000:
        b_sub = df_c[df_c['label'] == 0]
        benign_chunks.append(b_sub.head(5000 - len(benign_chunks)))
    for fam in attack_chunks:
        if len(attack_chunks[fam]) < 625:
            a_sub = df_c[df_c['attack_family'] == fam]
            if len(a_sub) > 0:
                attack_chunks[fam].append(a_sub.head(625 - len(attack_chunks[fam])))
    if len(benign_chunks) >= 5000 and all(sum(len(x) for x in attack_chunks[f]) >= 300 for f in attack_chunks):
        break

df_benign = pd.concat(benign_chunks, ignore_index=True)
dfs_attacks = [pd.concat(attack_chunks[f], ignore_index=True) for f in attack_chunks if len(attack_chunks[f]) > 0]
df_sample = pd.concat([df_benign] + dfs_attacks, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

X_sample = df_sample[FEATURE_COLS].to_numpy(dtype=np.float32)
y_sample = df_sample['label'].to_numpy(dtype=np.int8)
print(f'Total Evaluation Sample: {len(X_sample):,} flows (Benign: {np.sum(y_sample==0):,}, Attacks: {np.sum(y_sample==1):,})')

### Step 4: Initialize TreeExplainer for Tier 2 Gatekeeper

In [4]:
t2_explainer = shap.TreeExplainer(t2_booster)
print(f'Base value (log-odds): {t2_explainer.expected_value}')
t2_explanation = t2_explainer(X_sample)
t2_explanation.feature_names = FEATURE_COLS
print(f'Calculated SHAP matrix shape: {t2_explanation.values.shape}')

### Step 5: Global Feature Importance & Beeswarm Summary

In [5]:
plt.figure(figsize=(10, 6))
shap.plots.bar(t2_explanation, max_display=18)

plt.figure(figsize=(11, 7))
shap.plots.beeswarm(t2_explanation, max_display=18)

### Step 6: Tier 1 Behavioral Anomaly Outlier Attribution

In [6]:
t1_explainer = shap.TreeExplainer(t1_model)
X_scaled_eval = scaler.transform(X_sample[:2000])
t1_explanation = t1_explainer(X_scaled_eval)
t1_explanation.feature_names = FEATURE_COLS
t1_explanation.values = -t1_explanation.values

plt.figure(figsize=(10, 6))
shap.plots.bar(t1_explanation, max_display=18)

### Step 7: Local Single-Flow Waterfall Explanations

In [7]:
# Explain representative DDoS attack
ddos_idx = df_sample[df_sample['attack_family'] == 'DDoS'].index[0]
print(f'Explaining DDoS flow at index {ddos_idx}:')
shap.plots.waterfall(t2_explanation[ddos_idx], max_display=10)

### Step 8: Tabular Alert Attribution for Backend Integration

In [8]:
alert_flow = X_sample[ddos_idx]
alert_shap = t2_explanation.values[ddos_idx]
df_alert = pd.DataFrame({
    'feature': FEATURE_COLS,
    'observed_value': alert_flow,
    'shap_value': alert_shap,
    'abs_shap': np.abs(alert_shap),
    'direction': ['TOWARD_ATTACK' if s > 0 else 'TOWARD_BENIGN' for s in alert_shap]
}).sort_values('abs_shap', ascending=False).reset_index(drop=True)

display(df_alert.head(10))
df_alert.to_csv('../reports/shap/alert_feature_contributions.csv', index=False)

### Step 9: Mathematical Additive Verification & Audit Checklist

In [9]:
base_val = float(t2_explanation.base_values[ddos_idx])
sum_shap = float(np.sum(alert_shap))
reconstructed_margin = base_val + sum_shap
true_booster_margin = float(t2_booster.predict(X_sample[ddos_idx:ddos_idx+1], raw_score=True)[0])

print(f'Base value:           {base_val:+.4f}')
print(f'Sum of SHAP values:   {sum_shap:+.4f}')
print(f'Reconstructed margin: {reconstructed_margin:+.4f}')
print(f'Booster raw margin:   {true_booster_margin:+.4f}')
print(f'Discrepancy:          {abs(reconstructed_margin - true_booster_margin):.2e}')
assert abs(reconstructed_margin - true_booster_margin) < 1e-4, 'Additive verification failed!'